In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
from keras.datasets import imdb
from keras.utils import pad_sequences
print("Keras:", keras.__version__)

In [ ]:
VOCAB, MAXLEN = 10000, 200
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB)
x_train = pad_sequences(x_train, maxlen=MAXLEN)
x_test = pad_sequences(x_test, maxlen=MAXLEN)
print("Train:", x_train.shape, "| Test:", x_test.shape, "| Labels:", np.unique(y_train))

In [ ]:
def build(kind):
    cell = layers.LSTM(64) if kind == "LSTM" else layers.SimpleRNN(64)
    model = keras.Sequential([
        layers.Input(shape=(MAXLEN,)),
        layers.Embedding(VOCAB, 64),
        cell,
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ], name=kind)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

build("LSTM").summary()

In [ ]:
histories, scores = {}, {}
for kind in ["SimpleRNN", "LSTM"]:
    print(f"\n===== Training {kind} =====")
    m = build(kind)
    histories[kind] = m.fit(x_train, y_train, epochs=3, batch_size=128, validation_split=0.2, verbose=2)
    scores[kind] = m.evaluate(x_test, y_test, verbose=0)[1]
    globals()["model_" + kind] = m
print("\nTest accuracy:", {k: round(v, 4) for k, v in scores.items()})

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for kind, h in histories.items():
    ax[0].plot(h.history["val_accuracy"], label=kind)
    ax[1].plot(h.history["val_loss"], label=kind)
ax[0].set_title("Validation accuracy"); ax[1].set_title("Validation loss")
for a in ax: a.set_xlabel("epoch"); a.legend()
plt.show()

In [ ]:
word_index = imdb.get_word_index()
def encode(text):
    ids = [1]                                        # 1 = start token
    for w in text.lower().replace(".", "").replace(",", "").replace("!", "").split():
        i = word_index.get(w)
        ids.append(i + 3 if i is not None and i + 3 < VOCAB else 2)   # 2 = unknown
    return pad_sequences([ids], maxlen=MAXLEN)

for review in ["This movie was fantastic, I loved the acting and the story",
               "Terrible film, boring plot and awful acting, a waste of time"]:
    p = float(model_LSTM.predict(encode(review), verbose=0)[0][0])
    print(f"{'Positive' if p > 0.5 else 'Negative'} ({p:.2f}) <- {review}")